# KneeScan on Google Colab

Runs the KneeScan web app (ACL tear status and level, ACL location, ACL length and width, femur → fibular head distance) and gives you a public link for about a week, or for as long as this notebook keeps running.

**Before you start (once):**
1. **Runtime → Change runtime type → T4 GPU** (free). It also works on CPU, just slower (about 1.5–2 min per scan instead of about 20–40 s).
2. Click the **key icon** in the left sidebar (*Secrets*), add a secret named `HF_TOKEN` whose value is a Hugging Face **read** token for the account that owns `tariksyed/kneescan`, and switch on *Notebook access*.

Then **Runtime → Run all**. The last cell prints a `https://….gradio.live` link: open it, upload a scan, click **Analyze**.

*Research prototype, not for clinical use.*

In [ ]:
# 1. Check for a GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (CPU only, slower)')

In [ ]:
# 2. Install the libraries KneeScan needs (about 1-2 minutes)
!pip install -q transformers==5.22.1 monai==1.6.1 SimpleITK==2.5.7 pydicom==3.0.2 nibabel==5.4.2 scikit-image==0.26.0 gradio==5.50.0

In [ ]:
# 3. Download the KneeScan code and model weights from the private Hugging Face repo (about 470 MB)
import os
from huggingface_hub import snapshot_download
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    from getpass import getpass
    token = getpass('Hugging Face read token: ')
path = snapshot_download('tariksyed/kneescan', token=token, local_dir='/content/kneescan')
print('downloaded to', path)

In [ ]:
# 4. Start the app and print a public link (keep this cell running while you use it)
import os, sys
os.chdir('/content/kneescan'); sys.path.insert(0, '/content/kneescan')
import app
app.ENGINE.load()          # load models now so the first scan is quick
app.demo.queue(max_size=8).launch(share=True, debug=True)